In [ ]:
import os, glob
import pandas as pd

In [ ]:
files = glob.glob("../../data/regression_outputs/regmodels_seg/Fold/*/*/SV/Multi_segmentation/results.csv")
len(files)

In [ ]:
label_sdg_files = glob.glob("../../data/processed/0labels/*.csv")
labels_sdg_list = []
for label_sdg_file in label_sdg_files:
    labels_sdg = pd.read_csv(label_sdg_file)
    labels_sdg_list.append(labels_sdg)
labels_sdg = pd.concat(labels_sdg_list, ignore_index=True)
labels_sdg

In [ ]:
df_ls = []
for file in files:
    city_tmp = pd.read_csv(file)
    city_tmp['city'] = file.split('/')[-4]
    df_ls.append(city_tmp)
df = pd.concat(df_ls, ignore_index=True)
df

In [ ]:
df = df[df['r2'] > 0]
df = df[df['target'].isin(labels_sdg['ID'].unique())]
df.reset_index(drop=True, inplace=True)
df

In [ ]:
df['SDG'] = df['target'].map(dict(zip(labels_sdg['ID'], labels_sdg['SDG'])))

In [ ]:
output_file = "../../data/processed/fig/fig1b_sdg_r2_compare_token_detail.csv"
new_df = df[["city","target", "SDG",  "r2"]].copy()
new_df.rename(columns={"r2": "seg_r2"}, inplace=True)
if not os.path.exists(output_file):
    new_df.to_csv(output_file, index=False)
else:
    old_df = pd.read_csv(output_file)
    new_df.sort_values('city', inplace=True)
    old_df.sort_values('city', inplace=True)
    old_df['seg_r2'] = new_df['seg_r2']
    old_df.to_csv(output_file, index=False)

In [ ]:
del df['fold'], df['city'], df['target']
df

In [ ]:
sdg_df = df.groupby(['SDG']).mean().reset_index()
sdg_df

In [ ]:
sdg_df.columns = ['SDG', 'seg_mae', 'seg_mse', 'seg_r2']
sdg_df

In [ ]:
output_file = "../../data/processed/fig/fig1b_sdg_r2_compare_token.csv"
if not os.path.exists(output_file):
    sdg_df.to_csv(output_file, index=False)
else:
    old_df = pd.read_csv(output_file)
    sdg_df.sort_values('SDG', inplace=True)
    old_df.sort_values('SDG', inplace=True)
    old_df['seg_mae'] = sdg_df['seg_mae']
    old_df['seg_mse'] = sdg_df['seg_mse']
    old_df['seg_r2'] = sdg_df['seg_r2']
    old_df.to_csv(output_file, index=False)
    seg_df = old_df
seg_df